# 🤖 Model Training & Evaluation — NTB Predictive Model
**Trains:** Linear Regression, Random Forest, Gradient Boosting, XGBoost, LSTM
**Evaluated with:** 5-Fold TimeSeriesSplit CV (no data leakage)

In [ ]:
import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')
from src.data_loader import load_clean
from src.features    import engineer, get_feature_cols
from src.train       import train, predict_next
from src.evaluate    import metrics, plot_actual_vs_predicted, plot_model_comparison
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})
print('Imports OK')

## 1. Load & Prepare Data

In [ ]:
df      = load_clean('../data/raw/Primary_Market_in_Excel.xlsx')
feat364 = engineer(df, tenor=364)
fcols   = get_feature_cols(feat364)
print(f'Samples  : {feat364.shape[0]}')
print(f'Features : {len(fcols)}')
print(f'Date span: {feat364["auctionDate"].min().date()} → {feat364["auctionDate"].max().date()}')
feat364[fcols[:5] + ['target']].describe().round(4)

## 2. Train All Models (5-Fold TimeSeriesSplit)

In [ ]:
model, scaler, summary = train(feat364, fcols, n_splits=5)

## 3. Visual Model Comparison

In [ ]:
plot_model_comparison(summary, save_path='../data/processed/fig_model_comparison.png')

## 4. Hold-Out Evaluation (Last Fold)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing   import StandardScaler

tscv   = TimeSeriesSplit(n_splits=5)
X      = feat364[fcols]
y      = feat364['target']
splits = list(tscv.split(X))
tr_idx, te_idx = splits[-1]

sc = StandardScaler()
sc.fit_transform(X.iloc[tr_idx])
preds   = model.predict(sc.transform(X.iloc[te_idx]))
actuals = y.iloc[te_idx].values
dates   = feat364['auctionDate'].iloc[te_idx].values

result = metrics(actuals, preds, label='Random Forest — Hold-Out Fold')
print(result)

## 5. Actual vs Predicted Chart

In [ ]:
plot_actual_vs_predicted(
    actuals, preds, dates=dates,
    label='364-Day NTB — Random Forest',
    save_path='../data/processed/fig_actual_vs_pred.png'
)

## 6. Feature Importances

In [ ]:
imp = pd.Series(model.feature_importances_, index=fcols).nlargest(15).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp.index, imp.values, color='#1f77b4', alpha=0.85, edgecolor='white')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../data/processed/fig_feature_importance.png', dpi=150)
plt.show()
print(imp.round(4).to_string())

## 7. Next Auction Prediction

In [ ]:
pred      = predict_next(model, scaler, fcols, feat364)
last      = feat364['yield_pct'].iloc[-1]
last_date = feat364['auctionDate'].iloc[-1].strftime('%d %b %Y')
delta     = pred - last

print('=' * 45)
print('  NEXT AUCTION FORECAST — 364-Day NTB')
print('=' * 45)
print(f'  Last auction ({last_date}): {last:.4f}%')
print(f'  Predicted next yield     : {pred:.4f}%')
print(f'  Expected change          : {delta:+.4f}%')
print()
if abs(delta) < 0.10:
    print('  Signal: STABLE — monitor before deciding')
elif delta > 0:
    print('  Signal: RISING — consider investing now at current rate')
else:
    print('  Signal: FALLING — lock in current rate before next auction')

## 8. LSTM Model (Deep Learning Extension)

In [ ]:
from src.lstm_model import train_lstm

lstm_model, lstm_scaler, lstm_scores = train_lstm(
    df, tenor=364, lookback=12, epochs=60, batch_size=16,
    save_path='../models/lstm_model.keras'
)

print('\nLSTM vs Random Forest:')
print(f'  LSTM  — RMSE: {lstm_scores["RMSE"]:.4f}  R²: {lstm_scores["R2"]:.4f}')
rf_score = {k: round(float(__import__("numpy").mean(v)), 4) for k, v in summary["Random Forest"].items()}
print(f'  RF    — RMSE: {rf_score["RMSE"]:.4f}  R²: {rf_score["R2"]:.4f}')

## 9. Final Summary Table

In [ ]:
import numpy as np

rows = []
for name, m in summary.items():
    rows.append({
        'Model'  : name,
        'RMSE'   : round(float(np.mean(m['RMSE'])), 4),
        'MAE'    : round(float(np.mean(m['MAE'])),  4),
        'R²'     : round(float(np.mean(m['R2'])),   4),
        'CV Folds': '5'
    })
rows.append({
    'Model'  : 'LSTM',
    'RMSE'   : round(lstm_scores['RMSE'], 4),
    'MAE'    : round(lstm_scores['MAE'],  4),
    'R²'     : round(lstm_scores['R2'],   4),
    'CV Folds': 'Hold-out (80/20)'
})
result_df = pd.DataFrame(rows).sort_values('RMSE')
result_df['Best'] = result_df['RMSE'] == result_df['RMSE'].min()
print(result_df.to_string(index=False))